# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and process the FAIR² dataset on second primary colorectal cancer in survivors, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields by `@id`.

In [ ]:
# Fetch record set metadata from the main dataset

from pprint import pprint

record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in this dataset. Please check schema or dataset documentation.")
else:
    print(f"Found {len(record_sets)} record set(s) in the dataset:")
    for rs in record_sets:
        print(f"\nRecord Set @id: {rs['@id']}")
        print(f"Name: {rs.get('name', '')}")
        fields = rs.get('field', [])
        # Ensure fields is list
        if isinstance(fields, dict):
            fields = [fields]
        print("Fields:")
        for f in fields:
            if isinstance(f, dict):
                print(f"  - Field @id: {f['@id']}, name: {f.get('name','')}, dataType: {f.get('dataType','')}")
            else:
                print(f"  - Field @id: {f}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All field and record set references are by their `@id`.

_**Below, we'll first enumerate all available record sets by their `@id`, then extract all of them into Pandas DataFrames.**_

In [ ]:
# List all record sets (by @id)
record_set_ids = [r['@id'] for r in dataset.record_sets]

if not record_set_ids:
    print("No record sets available for extraction.")
else:
    print(f"Extracting {len(record_set_ids)} record set(s):")
    print(record_set_ids)

    # Load each record set into a dataframe and print columns
    dataframes = {}
    for rid in record_set_ids:
        try:
            # Records yields dictionaries for each record
            records = list(dataset.records(record_set=rid))
            df = pd.DataFrame(records)
            dataframes[rid] = df
            print(f"\nFirst 5 records for Record Set {rid} (columns):")
            print(df.columns.tolist())
            display(df.head())
        except Exception as e:
            print(f"Error extracting {rid}: {e}")

# For the rest of this notebook, select the first available record set (if any)
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    print(f"Selected record set for subsequent steps: {selected_record_set_id}")
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply basic EDA steps such as filtering records, normalizing numeric fields, and grouping data as appropriate. All references are by `@id`.

_**Note: Only applies if numeric fields are available in the selected record set.**_

In [ ]:
import numpy as np

if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    # Identify likely numeric fields: infer from dtype or field metadata if available
    # For this demonstration, check columns for typical identifiers
    numeric_candidates = [col for col in df.columns if df[col].dtype in [np.int64, np.float64, np.int32, np.float32]]
    # If none identified by dtype, try infer by column names containing certain keywords
    if not numeric_candidates:
        substrings = ['age', 'interval', 'duration', 'years', 'number', 'score', 'count']
        for col in df.columns:
            if any(s in col.lower() for s in substrings):
                # Try convert to numeric (errors='coerce' so non-numeric will become NaN)
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].notna().any():
                    numeric_candidates.append(col)
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Numeric field selected: {numeric_field_id}")
        # Filter on a threshold value (lets use 10 as a generic threshold)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try to group by another field
        # Use first categorical or object-type column that's not the numeric_field_id
        group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == 'object' and (df[col].nunique() < len(df)/2)]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found for EDA. Review data to identify numeric variables.")
else:
    print("No record set selected for EDA.")

## 5. Visualization
Visualize selected data distributions and relationships.

_**Below, we plot a histogram of the numeric field (if available) and a boxplot grouped by a category field.**_

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # Boxplot by a group field if available
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field visualized; not found or dataset empty.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² clinical oncology dataset using the Croissant metadata standard and the `mlcroissant` Python library.

- **Accessed dataset metadata and contents by their `@id` fields, ensuring traceability and reproducibility.**
- **Enumerated available record sets and fields; loaded records into pandas DataFrames for analysis.**
- **Explored data with basic EDA, including filtering, normalization, grouping, and visualization where possible.**

This workflow can be adapted for further statistical analysis or machine learning tasks on clinical datasets described with Croissant schemas.
